In [ ]:
import pandas as pd
df =pd.read_csv("GDP per capita (PPP).csv")
print('GDP per capita (PPP).csv\n', df.head())
print(df.columns)
print(df.describe())
print()
df =pd.read_csv("Natural Disaster.csv")
print('Natural Disaster.csv\n', df.head())
print(df.columns)
print(df.describe())

## Standardise & Merge

In [22]:
import pandas as pd
# Load dataset
df = pd.read_csv("Natural Disaster.csv")
# --- Extract Year (your previous steps) ---
df["DisNo."] = df["DisNo."].astype(str)
df["Year"] = df["DisNo."].str[:4]
df = df.drop(columns=["DisNo."])
# Reorder: Year + ISO
cols = ["Year", "ISO"] + [c for c in df.columns if c not in ["Year", "ISO"]]
df = df[cols]

# --- Create start and end date columns ---
# Convert start date fields to one column
df["startDate"] = (
    df["Start Year"].astype(str).str.zfill(4) + "-" +
    df["Start Month"].astype(str).str.zfill(2) + "-" +
    df["Start Day"].astype(str).str.zfill(2)
)
# Convert end date fields to one column
df["endDate"] = (
    df["End Year"].astype(str).str.zfill(4) + "-" +
    df["End Month"].astype(str).str.zfill(2) + "-" +
    df["End Day"].astype(str).str.zfill(2)
)
# --- Drop the original date components ---
df = df.drop(columns=[
    "Start Year", "Start Month", "Start Day",
    "End Year", "End Month", "End Day"
])
# Show result
print(df[["Year", "ISO", "startDate", "endDate"]].head())
df.to_csv('Natural Disaster_v2.csv', index=False)
df.shape

   Year  ISO      startDate        endDate
0  1960  BGD   1960-nan-nan   1960-nan-nan
1  1960  IRN  1960-4.0-24.0  1960-4.0-24.0
2  1960  PER  1960-1.0-13.0  1960-1.0-13.0
3  1960  NIU  1960-1.0-18.0  1960-1.0-18.0
4  1960  MAR  1960-2.0-29.0  1960-2.0-29.0


(16698, 26)

In [32]:
dis = pd.read_csv("Natural Disaster_v2.csv")  # your cleaned disaster dataset
gdp = pd.read_csv("GDP per capita (PPP).csv")

# Ensure keys have correct dtypes
dis["ISO"] = dis["ISO"].astype(str)
dis["Year"] = dis["Year"].astype(int)

gdp["Country Code"] = gdp["Country Code"].astype(str)
gdp["Year"] = gdp["Year"].astype(int)

# Merge: ISO ↔ Country Code, Year ↔ Year
merged = dis.merge(
    gdp[["Country Code", "Year", "GDP PPP (2021 international constant $)"]],
    left_on=["ISO", "Year"],
    right_on=["Country Code", "Year"],
    how="inner"   # << INNER JOIN as you requested
)
# Remove the redundant merge key
merged = merged.drop(columns=["Country Code"])
merged["coordinates"] = df.apply(
    lambda row: f"({row['Latitude']}, {row['Longitude']})", axis=1
)
# Save or display
print(merged.head())
print(merged.shape)
merged.to_csv('Natural Disaster_v3.csv', index=False)

   Year  ISO Disaster Subgroup      Disaster Type Disaster Subtype Event Name  \
0  1960  BGD      Hydrological              Flood  Flood (General)        NaN   
1  1960  IRN       Geophysical         Earthquake  Ground movement        NaN   
2  1960  PER       Geophysical         Earthquake  Ground movement        NaN   
3  1960  MAR       Geophysical         Earthquake  Ground movement        NaN   
4  1960  PNG       Geophysical  Volcanic activity         Ash fall      Manam   

     Region                                     Location Origin  \
0      Asia                                          NaN    NaN   
1      Asia                                  Lar, Gerash    NaN   
2  Americas  Arequipa, Chuquibamban Caravelli, Cotahuasi    NaN   
3    Africa                                       Agadir    NaN   
4   Oceania                                        Manam    NaN   

     Associated Types  ... Total Deaths No. Injured  No. Affected  \
0                 NaN  ...      10000.0  

In [40]:
pop = pd.read_csv("population_projections.csv")
print(pop.shape, "\n")
# ---- 1. unify the two population columns into one ----
pop["population"] = pop[
    "Population - Sex: all - Age: all - Variant: estimates"
].fillna(
    pop["Population - Sex: all - Age: all - Variant: medium"]
)
# Drop original two columns
pop = pop.drop(columns=[
    "Population - Sex: all - Age: all - Variant: estimates",
    "Population - Sex: all - Age: all - Variant: medium"
])
# ---- 2. remove projection years (keep only real data) ----
pop = pop[pop["Year"] <= 2024]
# Optional: reset index
pop = pop.reset_index(drop=True)

# Preview
print(pop.head())
print()
print(pop.info())
pop.to_csv('population_projections_v2.csv', index=False)

(38656, 4) 

  Code  Year  population
0  AFG  1950   7776180.0
1  AFG  1951   7879343.0
2  AFG  1952   7987784.0
3  AFG  1953   8096703.0
4  AFG  1954   8207954.0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19200 entries, 0 to 19199
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Code        17850 non-null  object 
 1   Year        19200 non-null  int64  
 2   population  19200 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 450.1+ KB
None


In [43]:
dis = pd.read_csv("Natural Disaster_v3.csv")
pop = pd.read_csv("population_projections_v2.csv")   # after your cleaning operations
# Ensure merge keys are the same type
dis["ISO"] = dis["ISO"].astype(str)
dis["Year"] = dis["Year"].astype(int)

pop["Code"] = pop["Code"].astype(str)
pop["Year"] = pop["Year"].astype(int)

# Perform inner join
merged_final = dis.merge(
    pop[["Code", "Year", "population"]],
    left_on=["ISO", "Year"],
    right_on=["Code", "Year"],
    how="inner"
)
# Remove duplicate country code column
merged_final = merged_final.drop(columns=["Code"])
# Preview
print(merged_final.shape)
print()
print(merged_final.head())
print()
print(merged_final.info())
merged_final.to_csv('Natural Disaster_v4.csv', index=False)

(16099, 29)

   Year  ISO Disaster Subgroup      Disaster Type Disaster Subtype Event Name  \
0  1960  BGD      Hydrological              Flood  Flood (General)        NaN   
1  1960  IRN       Geophysical         Earthquake  Ground movement        NaN   
2  1960  PER       Geophysical         Earthquake  Ground movement        NaN   
3  1960  MAR       Geophysical         Earthquake  Ground movement        NaN   
4  1960  PNG       Geophysical  Volcanic activity         Ash fall      Manam   

     Region                                     Location Origin  \
0      Asia                                          NaN    NaN   
1      Asia                                  Lar, Gerash    NaN   
2  Americas  Arequipa, Chuquibamban Caravelli, Cotahuasi    NaN   
3    Africa                                       Agadir    NaN   
4   Oceania                                        Manam    NaN   

     Associated Types  ... No. Injured No. Affected  No. Homeless  \
0                 NaN  ...  

In [48]:
df = pd.read_csv("Natural Disaster_v4.csv")

# Ensure numeric types
df["Total Deaths"] = pd.to_numeric(df["Total Deaths"], errors="coerce")
df["population"] = pd.to_numeric(df["population"], errors="coerce")

# Create affected_population column
df["affected_population"] = (df["Total Deaths"] / df["population"])*100

# Optional: handle cases where population = 0 or missing
df["affected_population"] = df["affected_population"].replace([float("inf")], float("nan"))

# Preview
print(df[["ISO", "Year", "Total Deaths", "population", "affected_population"]].head())
print()
print(df.shape)
# Save if needed
df.to_csv("DII_v5.csv", index=False)

   ISO  Year  Total Deaths  population  affected_population
0  BGD  1960       10000.0  51828663.0             0.019294
1  IRN  1960         480.0  21470433.0             0.002236
2  PER  1960          63.0  10174128.0             0.000619
3  MAR  1960       13100.0  11624932.0             0.112689
4  PNG  1960           NaN   1995115.0                  NaN

(16099, 30)


## Clean